In [1]:
import pandas as pd
import numpy as np


# Data Handling

In [2]:
url = "https://githubusercontent.com"
raw_df = pd.read_csv(url)

URLError: <urlopen error [Errno -5] No address associated with hostname>

In [ ]:
raw_df.head()

In [ ]:
raw_df.info()

In [ ]:
raw_df.isnull().sum()

In [ ]:
raw_df.duplicated().sum()

In [ ]:
raw_df.shape

In [ ]:
# Drop unwanted columns

new_df = raw_df.drop(['Booking_ID', 'date of reservation'], axis=1)

new_df.head()

In [ ]:
new_df.shape

In [ ]:
new_df.select_dtypes(include='object').columns

# Visualisation for finding Outliers and Correlation

## Outliers

In [ ]:
# checking for outliers in the numerical columns

import matplotlib.pyplot as plt

new_df.boxplot(figsize=(15,6), rot=90)

plt.title("Boxplots of Numerical Features")
plt.show()

In [ ]:
import seaborn as sns

sns.boxplot(y=new_df['average price'])

plt.title("Average Price Outliers")
plt.show()

In [ ]:
# counting the number of outliers

Q1 = new_df['average price'].quantile(0.25)
Q3 = new_df['average price'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = new_df[(new_df['average price'] < lower) |
              (new_df['average price'] > upper)]

print("Number of outliers:", len(outliers))

In [ ]:
# # keeping outliers for now until the models scores are found

## Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

new_df['type of meal'] = le.fit_transform(new_df['type of meal'])
new_df['room type'] = le.fit_transform(new_df['room type'])
new_df['market segment type'] = le.fit_transform(new_df['market segment type'])
new_df['booking status'] = le.fit_transform(new_df['booking status'])

# or use for loop for simpler code

# new_df = new_df.apply(
#     lambda col: LabelEncoder().fit_transform(col)
#     if col.dtype == 'object'
#     else col
# )

In [ ]:
new_df.dtypes

## Correlation

In [ ]:

plt.figure(figsize=(12,8))

sns.heatmap(
    new_df.corr(),
    annot=True,
    cmap='coolwarm'
)

plt.show()

In [ ]:
print(new_df.corr()['booking status'].sort_values(ascending=False))

In [ ]:
corr_booking = new_df.corr()['booking status'].drop('booking status')

corr_booking.sort_values().plot(
    kind='barh',
    figsize=(8,5)
)

plt.title('Correlation with Booking Status')
plt.show()

In [ ]:
# Strongest predictors for booking status is a negative corelation from lead time and positive corelation from special request

In [ ]:
print(new_df.corr()['average price'].sort_values(ascending=False))

In [ ]:
corr_price = new_df.corr()['average price'].drop('average price')

corr_price.sort_values().plot(
    kind='barh',
    figsize=(8,5)
)

plt.title('Correlation with Average Price')
plt.show()

In [ ]:
# Strongest predictors for average price are positive corelation from room type, market segment type, number of children, number of adults

# Random Forest Classifier (Booking Status)

## Splitting

In [ ]:
X = new_df.drop('booking status', axis=1)

y = new_df['booking status']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

## Model Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

In [ ]:
# n_estimators=20, max_depth=10

In [ ]:
rf.get_params(deep=True)

In [ ]:
y_pred = rf.predict(X_test)

## Evaluation (Accuracy, Confusion Matrix and Classification Report)

In [ ]:
# Accuracy

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

In [ ]:
# Confusion Matrix

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

plt.show()

In [ ]:
# Classification Report

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

# Multiple Linear Regression (Average Price)

## Splitting

In [ ]:
X_reg = new_df.drop('average price', axis=1)
y_reg = new_df['average price']

In [ ]:
from sklearn.model_selection import train_test_split

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train_reg.shape)
print(X_test_reg.shape)

print(y_train_reg.shape)
print(y_test_reg.shape)

## Model Training

In [ ]:
from sklearn.linear_model import LinearRegression

mlr = LinearRegression()

mlr.fit(X_train_reg, y_train_reg)

In [ ]:
y_pred_reg = mlr.predict(X_test_reg)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

r2 = r2_score(y_test_reg, y_pred_reg)
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

print("R2 Score:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
# low R2 - hence trying Random Forest Regressor for same train-test data

## Random Forest Regressor (Average Price)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(random_state=42)

rf_reg.fit(X_train_reg, y_train_reg)

In [ ]:
y_pred_rf = rf_reg.predict(X_test_reg)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

r2_rf = r2_score(y_test_reg, y_pred_rf)
mae_rf = mean_absolute_error(y_test_reg, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_reg, y_pred_rf))

print("R² Score:", r2_rf)
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)

In [ ]:
# Random Forest Regressor explains around 63.7% of the variation in average price (compared to only 38.8% for Multiple Linear Regression).
# MAE decreased from 20.27 to 13.28, hence predictions are closer to the actual prices.
# RMSE decreased from 27.25 to 21.00, indicating fewer large prediction errors.